# 09 — Data Drift Detection

QuPrep warns you when data passed to `transform()` is statistically different from the training distribution.

Two signals are checked per feature:
- **Mean shift** — flagged when `|new_mean - train_mean| / train_std > mean_threshold` (default 3σ)
- **Std ratio** — flagged when new std is more than `std_threshold`× the training std (default 2×)

```bash
pip install quprep
```

In [ ]:
import warnings

import numpy as np

import quprep as qd

rng = np.random.default_rng(42)
X_train   = rng.normal(loc=0.0,  scale=1.0, size=(200, 4))
X_same    = rng.normal(loc=0.0,  scale=1.0, size=(50,  4))
X_shifted = rng.normal(loc=10.0, scale=1.0, size=(50,  4))  # large mean shift

## 1. No drift — same distribution

In [ ]:
det = qd.DriftDetector(warn=False)
pipeline = qd.Pipeline(encoder=qd.AngleEncoder(), drift_detector=det)
pipeline.fit(X_train)
result = pipeline.transform(X_same)

print(result.drift_report)
print(f"overall_drift      : {result.drift_report.overall_drift}")
print(f"n_features_drifted : {result.drift_report.n_features_drifted}")

## 2. Drift detected — mean shifted by 10σ

In [ ]:
from quprep import QuPrepWarning

det2 = qd.DriftDetector(warn=True)
p2 = qd.Pipeline(encoder=qd.AngleEncoder(), drift_detector=det2)
p2.fit(X_train)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    result2 = p2.transform(X_shifted)

print(result2.drift_report)
print(f"drifted_features : {result2.drift_report.drifted_features}")
if caught:
    print(f"\nWarning: {caught[0].message}")

## 3. Per-feature statistics

In [ ]:
import pandas as pd

rows = []
for name, stats in result2.drift_report.feature_stats.items():
    rows.append({
        "feature":           name,
        "train_mean":        round(stats["train_mean"], 3),
        "new_mean":          round(stats["new_mean"], 3),
        "mean_shift_sigmas": round(stats["mean_shift_sigmas"], 2),
        "std_ratio":         round(stats["std_ratio"], 3),
        "drifted":           name in result2.drift_report.drifted_features,
    })

pd.DataFrame(rows)

## 4. Custom thresholds — more sensitive

In [ ]:
X_slight = rng.normal(loc=1.5, scale=1.0, size=(50, 4))  # subtle 1.5σ shift

det3 = qd.DriftDetector(mean_threshold=1.0, std_threshold=1.5, warn=False)
p3 = qd.Pipeline(encoder=qd.AngleEncoder(), drift_detector=det3)
p3.fit(X_train)
result3 = p3.transform(X_slight)

print(f"drift detected (tight 1σ) : {result3.drift_report.overall_drift}")
print(f"drifted_features          : {result3.drift_report.drifted_features}")

## 5. Standalone — no Pipeline

In [ ]:
from quprep.core.dataset import Dataset

ds_train = Dataset(data=X_train, feature_names=["age", "income", "score", "rate"])
ds_new   = Dataset(data=X_shifted, feature_names=["age", "income", "score", "rate"])

det4 = qd.DriftDetector(warn=False)
det4.fit(ds_train)
report = det4.check(ds_new)

print(report)

## 6. Suppress warning, check programmatically

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    result5 = p2.transform(X_shifted)

if result5.drift_report.overall_drift:
    print(f"Drift in: {result5.drift_report.drifted_features}")

## 7. Drift state preserved through save() / load()

In [ ]:
p2.save("/tmp/quprep_drift_pipeline.pkl")
loaded = qd.Pipeline.load("/tmp/quprep_drift_pipeline.pkl")

with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    result6 = loaded.transform(X_shifted)

print(f"overall_drift      : {result6.drift_report.overall_drift}")
print(f"n_features_drifted : {result6.drift_report.n_features_drifted}")
print("(same result as before serialization)")